# Round 16 · Audited turnover recovery

**Version: turnover-quarantine-20260912.** The original 2013 result is preserved; raw files and tournament labels are never edited. Round 17 already finished and must not be trained again.

This is a revised data-quality policy: paired games with impossible steal/turnover accounting are excluded from the two turnover-rate estimators only. The season exclusion limit is 1%; the seeded-team limit is 5%. These are safety limits, not tuned for Brier. If exceeded, stop and return the audit. Do not alter thresholds to force success.

Run `run_tests.py` in this folder before these cells. Outputs below are intentionally unexecuted.

In [ ]:
from pathlib import Path
import sys,json,subprocess
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display,FileLink
KIT=Path.home()/'march_research_release/repair_16_17'
sys.path.insert(0,str(KIT))
from run_round import run_stage
from round_plots import figures
pio.renderers.default='plotly_mimetype'
receipt=json.loads((KIT/'reports/test_receipt.json').read_text())
assert receipt['status']=='PASS','Run the user test gate first.'
print('Round 16 recovery; source and raw checks run inside each stage.')


## First: all-season, zero-fit audit
This catches the later-season inconsistency before the fitting loop. Detailed excluded-game rows stay private. A failed audit is a hard stop.

In [ ]:
run_stage('16','audit',max_seconds=120)
RUN=Path(json.loads((KIT/'reports/round16/latest_run.json').read_text())['run_dir'])
audit=pd.read_csv(RUN/'data_quality.csv');display(audit)
px.bar(audit,x='Season',y='excluded_fraction',title='Turnover evidence excluded — fraction of regular-season games').show()


## Reuse the accepted 2013 checkpoints
The migration checks original source, raw data, upstream provenance, checksums and that 2013 has zero exclusions. No fitting happens here. A conflict stops rather than overwriting.

In [ ]:
subprocess.run([sys.executable,str(KIT/'reuse_2013.py')],cwd=KIT,check=True,timeout=120)
run_stage('16','smoke',max_seconds=90)
run_stage('16','smoke',max_seconds=90)
s=json.loads((RUN/'smoke.json').read_text());display(s)
assert s['new_rating_fits']==0,'Unexpected repeated fitting; stop.'
print('READY_FOR_REMAINING_PREPARATION')


In [ ]:
run_stage('16','prepare',max_seconds=300)
display(json.loads((RUN/'prepare.json').read_text()))


In [ ]:
run_stage('16','evaluate',max_seconds=180)
display(pd.read_csv(RUN/'metrics.csv').round(7))
display(json.loads((RUN/'decisions.json').read_text()))
run_stage('16','report',max_seconds=120)


In [ ]:
for fig in figures(RUN,'16',KIT/'evidence'):fig.show()
report=KIT/'reports/round16/milestone_16_return.zip'
display(FileLink(str(report.relative_to(Path.cwd())) if report.is_relative_to(Path.cwd()) else str(report)))
print('Save this notebook with Ctrl+S. No round-17 refitting is necessary.')
